In [7]:
import torch
import sqlite3
import pandas as pd
from pathlib import Path
from PIL import Image
import time
from tqdm import tqdm

In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

BASE_DIR = Path("/home/agrupa-lab/agrupa")
DB_PATH = BASE_DIR / "agrupa.sqlite"
OUTPUT_DIR = Path("/home/agrupa-lab/agrupa/IE_capstones/Omar/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "deepseek_descriptions.csv"

Device: cuda
GPU: NVIDIA GeForce RTX 5090
VRAM: 33.7 GB


In [9]:
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT a.cat_no, a.titulo, a.autor, a.is_fauna, a.is_religious, a.century,
           i.file_path
    FROM artwork a
    INNER JOIN artwork_image i ON a.cat_no = i.cat_no
    WHERE substr(a.cat_no, 1, 1) = 'P'
"""
df = pd.read_sql(query, conn)
conn.close()

df['image_exists'] = df['file_path'].apply(lambda x: Path(x).exists())
df_ready = df[df['image_exists']].reset_index(drop=True)
print(f"Ready for DeepSeek processing: {len(df_ready)}")

Ready for DeepSeek processing: 6266


In [10]:
import subprocess
result = subprocess.run(['pip', 'show', 'deepseek-vl'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Name: deepseek_vl
Version: 1.0.0
Summary: DeepSeek-VL
Home-page: 
Author: DeepSeek-AI
Author-email: 
License: MIT License

Copyright (c) 2023 DeepSeek

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES 

In [11]:
# NOTE: Run this in terminal first if not installed:
# pip install deepseek-vl

from deepseek_vl.models import VLChatProcessor, MultiModalityCausalLM
from deepseek_vl.utils.io import load_pil_images

print("Loading DeepSeek-VL model...")

MODEL_NAME = "deepseek-ai/deepseek-vl-7b-chat"

vl_chat_processor = VLChatProcessor.from_pretrained(MODEL_NAME)
tokenizer = vl_chat_processor.tokenizer

model = MultiModalityCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE
)
model = model.to(DEVICE)
model.eval()

print(f"Model loaded! VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Python version is above 3.10, patching the collections module.
Loading DeepSeek-VL model...


processor_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

The image processor of type `VLMImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
PROMPT = "Describe this artwork in detail, including the people or animals depicted, their appearance, actions, social roles, and the overall mood of the scene."

def generate_description(image_path):
    try:
        conversation = [
            {
                "role": "User",
                "content": f"<image_placeholder>{PROMPT}",
                "images": [str(image_path)]
            },
            {
                "role": "Assistant",
                "content": ""
            }
        ]
        
        pil_images = load_pil_images(conversation)
        
        prepare_inputs = vl_chat_processor(
            conversations=conversation,
            images=pil_images,
            force_batchify=True
        ).to(DEVICE)
        
        inputs_embeds = model.prepare_inputs_embeds(**prepare_inputs)
        
        with torch.no_grad():
            outputs = model.language_model.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=prepare_inputs.attention_mask,
                pad_token_id=tokenizer.eos_token_id,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                max_new_tokens=300,
                do_sample=False
            )
        
        description = tokenizer.decode(
            outputs[0].cpu().tolist(),
            skip_special_tokens=True
        )
        return description.strip()
    
    except Exception as e:
        return f"ERROR: {str(e)}"

In [ ]:
sample = df_ready.sample(5, random_state=42)

print("Running DeepSeek smoke test on 5 artworks...\n")
for _, row in sample.iterrows():
    desc = generate_description(row['file_path'])
    print(f"Artwork: {row['titulo']}")
    print(f"Author:  {row['autor']}")
    print(f"Description: {desc}")
    print("-" * 60)